# Topic 3: Introduction to Machine Learning using Python and Scikit-learn

This student-ready notebook uses the mobile-money transaction dataset in `Data/mobile_money_statements.csv` to build a binary fraud-classification model.

## Learning outcomes

By the end of this practical, you should be able to:

- identify the roles of NumPy, Pandas, Matplotlib and Scikit-learn;
- load, inspect and prepare a real-world dataset;
- distinguish features from a target variable;
- handle missing values and categorical variables;
- create and scale useful features;
- split data into training and testing sets;
- select, train and use a basic Logistic Regression model; and
- evaluate and interpret model predictions.

> **Important:** `manual_review_score` and `settlement_status` are post-outcome fields in this dataset. They are excluded from the predictive features because using them would leak information from the future into the model.

## Frame the problem before coding

Following the Week 1 ML lifecycle, start with the question rather than the algorithm:

- **Business objective:** identify suspicious mobile-money transactions for review.
- **ML task:** supervised binary classification because `is_fraud` has two classes.
- **Prediction time:** use only information available when the transaction is scored.
- **Success measure:** compare a model with a simple baseline using ROC-AUC, precision and recall; accuracy alone can be misleading when fraud is uncommon.
- **Constraints:** avoid target leakage, preserve an honest time order, and keep the workflow reproducible.

The complete lifecycle is iterative: **frame → get and explore data → prepare → train → evaluate → deploy → monitor → reframe**. This notebook demonstrates the first five steps.

## Why the split and baseline matter

For financial transactions, a random split can allow the model to learn from the future. We therefore sort by transaction time and train on earlier transactions while testing on later transactions. The test set remains sealed until final evaluation.

A majority-class baseline is the simplest useful reference: it always predicts the more common class. A model should be judged by the improvement it provides over this baseline, not by its score in isolation.

## 3.1 Machine-learning libraries

- **NumPy** provides fast numerical arrays and mathematical operations.
- **Pandas** provides labelled tables (`DataFrame`) for loading and transforming data.
- **Matplotlib** creates plots for understanding distributions and model results.
- **Scikit-learn** provides preprocessing tools, estimators, train/test splitting and evaluation metrics.

The next cell imports the libraries used in this practical.

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

%matplotlib inline
pd.set_option("display.max_columns", 40)
RANDOM_STATE = 42

## Load and explore the data

The path below is relative to the repository root. If you open this notebook from another directory, update `DATA_PATH` accordingly.

In [ ]:
DATA_PATH = "Data/mobile_money_statements.csv"
raw = pd.read_csv(DATA_PATH)
print(f"Rows: {raw.shape[0]:,} | Columns: {raw.shape[1]}")
display(raw.head())
raw.info()

In [ ]:
# Explore column types, missing values and the target distribution.
summary = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing_count": raw.isna().sum(),
    "missing_percent": (raw.isna().mean() * 100).round(2),
})
display(summary)
display(raw["is_fraud"].value_counts(normalize=True).rename("proportion"))

## 3.2 Prepare the data

### Identify the target and create features

`is_fraud` is the **target** (`y`) we want to predict. The remaining selected columns are **features** (`X`). Identifiers such as `txn_id`, names and phone numbers are excluded because they identify records rather than describe a transaction. The two post-outcome fields are also excluded to prevent target leakage.

In [ ]:
def parse_amount(value):
    """Convert values such as '1,329/- Dr' or '(227/-)' to signed numbers."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    negative = text.startswith("(") or re.search(r"\bCr\b", text, flags=re.IGNORECASE)
    number = re.sub(r"[^0-9.]", "", text)
    if not number:
        return np.nan
    amount = float(number)
    return -amount if negative else amount


data = raw.copy()
data["txn_timestamp"] = pd.to_datetime(data["txn_time"], errors="coerce")
data["amount_signed"] = data["amount"].map(parse_amount)
data["amount_abs"] = data["amount_signed"].abs()
data["transaction_hour"] = data["txn_timestamp"].dt.hour
data["transaction_day"] = data["txn_timestamp"].dt.dayofweek
data["is_weekend"] = data["transaction_day"].isin([5, 6]).astype(int)

print(f"Unparsed timestamps: {data['txn_timestamp'].isna().sum()}")
print(f"Unparsed amounts: {data['amount_signed'].isna().sum()}")

### Feature engineering and train/test split

The engineered amount and time features make the raw transaction fields usable by a model. We use `stratify=y` so both sets retain approximately the same fraud proportion. The test set is held back until final evaluation.

In [ ]:
feature_columns = [
    "amount_abs",
    "balance_after",
    "gps_lat",
    "gps_lon",
    "transaction_hour",
    "transaction_day",
    "is_weekend",
    "txn_type",
    "region",
    "segment",
]
target_column = "is_fraud"

modeling_data = data.dropna(subset=["txn_timestamp"]).sort_values("txn_timestamp")
X = modeling_data[feature_columns]
y = modeling_data[target_column]

split_index = int(len(modeling_data) * 0.80)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Training rows: {len(X_train):,}")
print(f"Testing rows:  {len(X_test):,}")
print(f"Training period: {modeling_data["txn_timestamp"].iloc[0]} to {modeling_data["txn_timestamp"].iloc[split_index - 1]}")
print(f"Testing period:  {modeling_data["txn_timestamp"].iloc[split_index]} to {modeling_data["txn_timestamp"].iloc[-1]}")
print(f"Training fraud rate: {y_train.mean():.3f}")
print(f"Testing fraud rate:  {y_test.mean():.3f}")

### Imputation, encoding and feature scaling

- Numeric missing values are replaced with the training median and then standardised.
- Categorical missing values are replaced with `Unknown` and encoded as one-hot columns.
- A `ColumnTransformer` applies the correct preparation to each column type.
- A `Pipeline` keeps preparation inside the training process, preventing test-set information from influencing the fitted transformations.

In [ ]:
numeric_features = [
    "amount_abs",
    "balance_after",
    "gps_lat",
    "gps_lon",
    "transaction_hour",
    "transaction_day",
    "is_weekend",
]
categorical_features = ["txn_type", "region", "segment"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

## 3.3 Build a basic machine-learning model

Logistic Regression is an appropriate first classification algorithm because the target has two classes and the model is relatively easy to interpret. The model is initialised, trained with `.fit()`, and then used to generate predictions with `.predict()`.

In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

# Training happens here: preprocessing and model fitting use X_train only.
model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

print("First five predicted classes:", predictions[:5])
print("First five fraud probabilities:", probabilities[:5].round(3))

### Establish a majority-class baseline

The baseline predicts the most common training class for every test transaction. It is intentionally simple, but it gives us a floor for comparison.

In [ ]:
majority_class = y_train.mode().iloc[0]
baseline_predictions = np.full(len(y_test), majority_class)
print(f"Baseline class: {majority_class}")
print(f"Baseline accuracy: {accuracy_score(y_test, baseline_predictions):.3f}")
print(classification_report(y_test, baseline_predictions, target_names=["not fraud", "fraud"], zero_division=0))

### Evaluate model performance

Accuracy measures the fraction of correct class labels. Precision, recall and F1-score provide more detail for the fraud class. ROC-AUC evaluates how well the probability scores rank fraud cases above non-fraud cases.

In [ ]:
print(f"Accuracy: {accuracy_score(y_test, predictions):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, probabilities):.3f}")
print("\nClassification report:")
print(classification_report(y_test, predictions, target_names=["not fraud", "fraud"]))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=["not fraud", "fraud"],
    cmap="Blues",
)
plt.title("Test-set confusion matrix")
plt.show()

### Interpret predictions

A probability is not a guarantee: it is the model's estimated likelihood under the patterns learned from the training data. The following table compares actual labels, predicted labels and probabilities for individual test transactions.

In [ ]:
examples = X_test.head(10).copy()
examples["actual_fraud"] = y_test.loc[examples.index]
examples["predicted_fraud"] = predictions[:10]
examples["fraud_probability"] = probabilities[:10].round(3)
display(examples[["amount_abs", "txn_type", "region", "actual_fraud", "predicted_fraud", "fraud_probability"]])

In [ ]:
# Inspect the strongest linear-model effects after preprocessing.
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]
effects = pd.Series(coefficients, index=feature_names).sort_values()

print("Features most associated with the not-fraud class:")
display(effects.head(10))
print("Features most associated with the fraud class:")
display(effects.tail(10).sort_values(ascending=False))

## Alternative target: `monthly_inflow_kes` (regression)

If the target is a numeric amount such as `monthly_inflow_kes`, the problem changes from **classification** to **regression**. Instead of predicting a class label, the model predicts a continuous value. The target is not a raw column in this dataset, so we derive it as the monthly sum of `receive` and `cashin` amounts for each customer.

This example uses previous-month activity as input features and the current month's inflow as the target. Keeping predictors from before the target period is important for a realistic forecast.

In [ ]:
monthly_source = data.copy()
monthly_source["month"] = monthly_source["txn_timestamp"].dt.to_period("M")

inflow_types = ["receive", "cashin"]
monthly_source["inflow_amount"] = monthly_source["amount_abs"].where(
    monthly_source["txn_type"].isin(inflow_types), 0
)

monthly = (
    monthly_source.groupby(["reg_id", "month"], as_index=False)
    .agg(
        monthly_inflow_kes=("inflow_amount", "sum"),
        transaction_count=("txn_id", "count"),
        average_balance=("balance_after", "mean"),
        region=("region", "first"),
        segment=("segment", "first"),
    )
    .sort_values(["reg_id", "month"])
)

for column in ["monthly_inflow_kes", "transaction_count", "average_balance"]:
    monthly[f"previous_{column}"] = monthly.groupby("reg_id")[column].shift(1)

regression_data = monthly.dropna(
    subset=["previous_monthly_inflow_kes", "previous_transaction_count", "previous_average_balance"]
).copy()
display(regression_data.head())
print(f"Customer-month rows: {len(regression_data):,}")

### Build and evaluate a regression model

For regression, use metrics designed for numeric errors: **MAE** is the average absolute error in KES, **RMSE** penalises larger errors more strongly, and **R²** measures the proportion of target variation explained by the model.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

regression_numeric = [
    "previous_monthly_inflow_kes",
    "previous_transaction_count",
    "previous_average_balance",
]
regression_categorical = ["region", "segment"]
regression_features = regression_numeric + regression_categorical

X_reg = regression_data[regression_features]
y_reg = regression_data["monthly_inflow_kes"]
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_STATE
)

regression_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), regression_numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore")),
    ]), regression_categorical),
])

regression_model = Pipeline([
    ("preprocessor", regression_preprocessor),
    ("regressor", LinearRegression()),
])
regression_model.fit(X_reg_train, y_reg_train)
inflow_predictions = regression_model.predict(X_reg_test)

rmse = np.sqrt(mean_squared_error(y_reg_test, inflow_predictions))
print(f"MAE:  {mean_absolute_error(y_reg_test, inflow_predictions):,.2f} KES")
print(f"RMSE: {rmse:,.2f} KES")
print(f"R²:   {r2_score(y_reg_test, inflow_predictions):.3f}")

The original fraud example remains a classification problem because `is_fraud` is a 0/1 class label. The `monthly_inflow_kes` example is a regression problem because its target is a continuous numeric amount. Choose the estimator, predictions and metrics to match the target type.

## Practical exercise

Fork this repository and work on a branch of your own. Then extend this notebook:

1. Plot the distribution of `amount_abs` for fraud and non-fraud transactions.
2. Add one defensible feature, such as a missing-GPS indicator or a log-transformed amount, and compare ROC-AUC.
3. Train a Decision Tree or Random Forest in a second pipeline.
4. Compare the models using accuracy, recall, F1-score and ROC-AUC.
5. Choose a probability threshold other than `0.5` and explain how it changes fraud recall and false positives.

Do not add `manual_review_score` or `settlement_status` to the predictive features: both are recorded after the outcome and would make the evaluation misleading.